# Neural nets — when a straight line is not enough

Module 2's classifiers draw **straight** (or piecewise-straight) decision boundaries.
A neural network can **curve**. This notebook puts that claim on trial.

We generate two interlocking rings of points (`make_circles`). No straight line can
separate them. Then we train:

- **Logistic Regression** — the linear classifier (Module 2's ancestor)
- **MLP** — a tiny multilayer perceptron with two hidden layers `(32, 16)`

Press play on each cell. ▶

In [ ]:
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt

X, y = make_circles(n_samples=1000, noise=0.15, factor=0.5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"{len(X_train)} points to train on, {len(X_test)} held out to test")

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=12, alpha=0.8)
plt.title("Two interlocking rings — no straight line separates them")
plt.axis("equal"); plt.show()

### Train both models, score on unseen points

Accuracy = fraction of points classified correctly. 0.5 is a coin flip on a balanced
two-class problem. The number that matters is **test** accuracy.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "MLP (32 → 16)":       MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42),
}

print(f"{'model':>22} | {'train acc':>9} | {'test acc':>8}")
print("-" * 46)
fitted = {}
for name, m in models.items():
    pipe = make_pipeline(StandardScaler(), m)
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    print(f"{name:>22} | {pipe.score(X_train, y_train):>9.3f} | {pipe.score(X_test, y_test):>8.3f}")

### Picture the decision boundaries

Logistic regression draws a **line**. The MLP draws a **ring-shaped** boundary that
actually fits the data. That gap — line vs curve — is why we stack neurons with
non-linear activations.

In [ ]:
import numpy as np

xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 300), np.linspace(-1.5, 1.5, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, (name, pipe) in zip(axes, fitted.items()):
    zz = pipe.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, cmap="coolwarm", alpha=0.35)
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", s=14, edgecolors="k", linewidths=0.2)
    ax.set_title(f"{name}\ntest acc = {pipe.score(X_test, y_test):.3f}")
    ax.set_aspect("equal")
plt.tight_layout(); plt.show()

### The takeaway

Logistic regression lands near a coin flip (~0.56 test). The tiny MLP clears 0.90+.
Same training data, same evaluation — the only difference is the **curved** boundary
the network can invent by stacking non-linear neurons.

That is deep learning in miniature. Scale the same idea (weights, activations, loss,
backprop) to images, text, and audio — and you are looking at the engine behind modern AI.

**Try it yourself:** change `noise=0.15` to `noise=0.35` and re-run. As the rings get
messier, both models get harder — watch how much of the MLP's lead survives.